# RAVDESS: Emotion label extraction & dataset verification
This notebook converts the script `reading.py` into notebook cells. Update `DATASET_PATH` if needed, then run cells sequentially.

In [1]:
import os
import glob
import pandas as pd

# ============================================================
# TASK 1: Read all 1440 audio files (sorted for reproducibility)
# ============================================================
# Update this to your dataset folder if different
DATASET_PATH = "/Users/devanshbansal/Desktop/ser/dataset/Audio_Speech_Actors_01-24"

audio_files = sorted(
    glob.glob(os.path.join(DATASET_PATH, "**", "*.wav"), recursive=True)
)

print(f"Total audio files found: {len(audio_files)}")
# assert len(audio_files) == 1440, "Expected 1440 files, check your dataset path!"


Total audio files found: 1440


In [2]:
# ============================================================
# TASK 2: Extract full metadata from each filename
# RAVDESS filename format (7 numeric identifiers separated by '-'):
# modality-vocal_channel-emotion-intensity-statement-repetition-actor.wav
# Example: 03-01-06-01-02-01-12.wav

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

intensity_map = {
    "01": "normal",
    "02": "strong"
}

statement_map = {
    "01": "Kids are talking by the door",
    "02": "Dogs are sitting by the door"
}
def merge_emotions(emotion):

    if emotion == "neutral":
        return "calm"

    return emotion

def extract_metadata(file_path):
    filename = os.path.basename(file_path)
    name_no_ext = filename.replace(".wav", "")
    parts = name_no_ext.split("-")

    if len(parts) != 7:
        return None  # skip malformed filenames

    modality_code, vocal_channel_code, emotion_code, intensity_code, \
        statement_code, repetition_code, actor_code = parts

    actor_num = int(actor_code)
    gender = "female" if actor_num % 2 == 0 else "male"

    return {
        "file_path": file_path,
        "filename": filename,
        "emotion": emotion_map.get(emotion_code, "unknown"),
        "emotion_code": emotion_code,
        "actor": actor_num,
        "gender": gender,
        "intensity": intensity_map.get(intensity_code, "unknown"),
        "statement": int(statement_code),
        "statement_text": statement_map.get(statement_code, "unknown"),
        "repetition": int(repetition_code),
        "vocal_channel": "speech" if vocal_channel_code == "01" else "song",
        "modality": {"01": "full_AV", "02": "video_only", "03": "audio_only"}.get(modality_code, "unknown")
    }



In [3]:
# ============================================================
# TASK 3: Create a DataFrame
# ============================================================
records = [extract_metadata(fp) for fp in audio_files]
records = [r for r in records if r is not None]  # drop malformed entries

df = pd.DataFrame(records)

# Reorder columns to match the recommended layout
df = df[[
    "file_path", "filename", "emotion", "emotion_code",
    "actor", "gender", "intensity", "statement",
    "statement_text", "repetition", "vocal_channel", "modality"
]]

print("\nSample rows:")
print(df.head())
print(f"\nDataFrame shape: {df.shape}")

df["emotion"] = df["emotion"].apply(
    merge_emotions
)
print(df["emotion"].unique())


Sample rows:
                                           file_path  \
0  /Users/devanshbansal/Desktop/ser/dataset/Audio...   
1  /Users/devanshbansal/Desktop/ser/dataset/Audio...   
2  /Users/devanshbansal/Desktop/ser/dataset/Audio...   
3  /Users/devanshbansal/Desktop/ser/dataset/Audio...   
4  /Users/devanshbansal/Desktop/ser/dataset/Audio...   

                   filename  emotion emotion_code  actor gender intensity  \
0  03-01-01-01-01-01-01.wav  neutral           01      1   male    normal   
1  03-01-01-01-01-02-01.wav  neutral           01      1   male    normal   
2  03-01-01-01-02-01-01.wav  neutral           01      1   male    normal   
3  03-01-01-01-02-02-01.wav  neutral           01      1   male    normal   
4  03-01-02-01-01-01-01.wav     calm           02      1   male    normal   

   statement                statement_text  repetition vocal_channel  \
0          1  Kids are talking by the door           1        speech   
1          1  Kids are talking by the door

In [4]:
# ============================================================
# TASK 4: Verify the dataset
# ============================================================
print("\n--- Verification ---")

# Row count
print("Total rows:", len(df))
# assert len(df) == 1440, "Row count mismatch!"

# Unknown/missing emotion check
print("Unknown emotion count:", (df["emotion"] == "unknown").sum())

# Unique emotion labels (should be exactly 8, alphabetically listed)
print("\nUnique emotions:")
print(sorted(df["emotion"].unique()))

# Emotion distribution (in official RAVDESS order, not frequency order)
emotion_order = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised"
]

print("\nEmotion distribution:")
print(df["emotion"].value_counts().reindex(emotion_order))

# Gender distribution
print("\nGender distribution:")
print(df["gender"].value_counts())

# Intensity distribution
print("\nIntensity distribution:")
print(df["intensity"].value_counts())

# Duplicate file check
print("\nDuplicate files:", df.duplicated("file_path").sum())

# Statement balance check (should be 720 / 720)
print("\nStatement distribution:")
print(df["statement_text"].value_counts())

# Actor list and count
print("\nActors present:", sorted(df["actor"].unique()))
print("Number of unique actors:", df["actor"].nunique())

# Folder/actor consistency check (every actor should have 60 files)
print("\nFiles per actor:")
print(df.groupby("actor").size().sort_index())

incomplete_actors = df.groupby("actor").size().sort_index()
incomplete_actors = incomplete_actors[incomplete_actors != 60]
if len(incomplete_actors) > 0:
    print("\nWARNING: Actors with incomplete file counts:")
    print(incomplete_actors)
else:
    print("\nAll actors have exactly 60 files. Dataset is complete.")

# Null value check
print("\nMissing values:\n", df.isnull().sum())

# Confirm files actually exist on disk
missing_files = df[~df["file_path"].apply(os.path.exists)]
print(f"\nMissing files on disk: {len(missing_files)}")


--- Verification ---
Total rows: 1440
Unknown emotion count: 0

Unique emotions:
['angry', 'calm', 'disgust', 'fearful', 'happy', 'sad', 'surprised']

Emotion distribution:
emotion
neutral        NaN
calm         288.0
happy        192.0
sad          192.0
angry        192.0
fearful      192.0
disgust      192.0
surprised    192.0
Name: count, dtype: float64

Gender distribution:
gender
male      720
female    720
Name: count, dtype: int64

Intensity distribution:
intensity
normal    768
strong    672
Name: count, dtype: int64

Duplicate files: 0

Statement distribution:
statement_text
Kids are talking by the door    720
Dogs are sitting by the door    720
Name: count, dtype: int64

Actors present: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Number of unique actors: 24

Files per actor:
actor
1     60
2     60
3     60
4     60
5     60
6     60
7     60
8     60
9     60
10    60
11    60
12    60
13    60
14    60
15    60
16    60
17    6

In [5]:
# ============================================================
# TASK 5: Save for next stages
# ============================================================
OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'outputs'))  # adjust as needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "ravdess_dataset.csv")
OUTPUT_PKL = os.path.join(OUTPUT_DIR, "ravdess_dataset.pkl")

df.to_csv(OUTPUT_CSV, index=False)
df.to_pickle(OUTPUT_PKL)  # preserves dtypes for later stages

print(f"\nSaved dataset to '{OUTPUT_CSV}' and '{OUTPUT_PKL}'")


Saved dataset to '/Users/devanshbansal/Desktop/ser/outputs/ravdess_dataset.csv' and '/Users/devanshbansal/Desktop/ser/outputs/ravdess_dataset.pkl'


In [6]:
# ============================================================
# COMPLETION SUMMARY
# ============================================================
print('\n' + '=' * 60)
print("DATASET VERIFICATION COMPLETED SUCCESSFULLY")
print('=' * 60)
print(f"Total Samples   : {len(df)}")
print(f"Unique Actors   : {df['actor'].nunique()}")
print(f"Unique Emotions : {df['emotion'].nunique()}")
print("Dataset Status  : READY FOR PREPROCESSING")
print('=' * 60)


DATASET VERIFICATION COMPLETED SUCCESSFULLY
Total Samples   : 1440
Unique Actors   : 24
Unique Emotions : 7
Dataset Status  : READY FOR PREPROCESSING
